# Глава 14. Компьютерное зрение — практика в Jupyter

Те же задачи, что на сайте курса, но в своём Jupyter: так вы привыкаете к блокноту, горячим клавишам и своему окружению. Проверка — та же, что в браузере.

**Как запустить** (один раз поставьте [uv](https://docs.astral.sh/uv/), см. урок «Своё рабочее место — uv и Jupyter» главы 2). В терминале, в папке курса:

```sh
uv run --with-requirements site/notebooks/requirements.txt jupyter lab site/notebooks
```

Без uv: `python3 -m venv .venv`, затем `.venv/bin/pip install -r site/notebooks/requirements.txt` (в Windows — `.venv\Scripts\pip`) и `.venv/bin/jupyter lab site/notebooks`. Откройте этот файл в папке `site/notebooks` — рядом с ним лежит расширение `dsc.py`, без него проверка не заработает.

**Как решать**

1. Запустите ячейку ниже с `%load_ext dsc` — она подключает проверку. После перезапуска ядра запустите её снова.
2. В ячейке задачи первая строка — `%%task глава/урок/id`. Её не трогайте, пишите код ниже и запускайте **Shift+Enter**. Под выводом появится «✓ Верно» или подсказка, что не так.
3. Переменные живут между ячейками, как в любом блокноте. Исключение — переменные, в которые задача просит положить ответ: перед запуском ячейки они стираются, чтобы проверка видела результат именно этого запуска. Подготовка задачи (если она есть) выполняется перед вашим кодом при каждом запуске.
4. Застряли — `dsc.hint('id')` покажет следующую подсказку, `dsc.solution('id', sure=True)` — эталон с разбором. `dsc.progress()` — что решено в этом сеансе.
5. Файлы данных берутся по имени, как на сайте: `pd.read_csv('orders.csv')`. Код работает в папке `site/notebooks/_work` — там копия `site/data`, и туда же попадает всё, что сохраняет ваш код.

Прогресс сайта и блокнота не связаны: отметки о решённых задачах на сайте ставятся только там. Чтобы начать с чистого листа, скачайте блокнот заново или возьмите его из архива курса.

In [ ]:
%load_ext dsc

## Урок 1. Картинка — это массив чисел

Над витриной «Зерна» висит камера: если товар закончился, его нужно вовремя допечь. Чтобы научить компьютер видеть пустую ячейку, сначала разбираемся, что такое изображение для Python: таблица чисел-яркостей. Смотрим на кадры, режем, отражаем, считаем простые признаки — и видим, почему сравнивать картинки «пиксель к пикселю» плохо.

Теория — на сайте курса: `ch14/01-pixels.html`. Задачи «собери код из строк» (1) остались на сайте.

### Что в файле · разминка

Загрузите `showcase.npz` и сохраните `X`, `y`, `classes`, а также `counts` — словарь `{название класса: число картинок}` и `size` — кортеж `(высота, ширина)` одной картинки.

*Подсказки (3): `dsc.hint('ch14/01-pixels/load')`*

In [ ]:
%%task ch14/01-pixels/load
import numpy as np

### Центр всех кадров · разминка

Вырежьте центральный квадрат 16 × 16 (строки и столбцы 8…23) у **всех** картинок сразу — одной строкой, без цикла. Результат — `centers`. В `n_numbers` — сколько всего чисел в `centers`, в `share_kept` — какая доля чисел исходного `X` осталась (2 знака).

*Подсказки (2): `dsc.hint('crop')`*

In [ ]:
%%task ch14/01-pixels/crop
import numpy as np

### Сделать ярче без поломки · база

Напишите функцию `brighten(img, k)`, которая умножает яркость картинки `uint8` на `k`, обрезает значения до 0–255, округляет и возвращает снова `uint8`. Примените её к `X[0]` с `k=1.5` — результат в `bright`. В `n_sat` — сколько пикселей `bright` стали ровно 255.

*Подсказки (3): `dsc.hint('overflow')`*

In [ ]:
%%task ch14/01-pixels/overflow
import numpy as np

def brighten(img, k):
    ...

### Правило «есть ли товар» · база

Посчитайте для каждой картинки `dark` — долю пикселей темнее 100. Правило: «товар есть», если `dark >= t`. Переберите пороги `[0.005, 0.01, 0.02, 0.05]` и сохраните `acc_by_t` — словарь `{порог: доля верных ответов}` (3 знака) для задачи «пусто / не пусто», `best_t` — лучший порог и `dark_median` — Series медиан `dark` по названиям классов (3 знака).

*Подсказки (3): `dsc.hint('dark-rule')`*

In [ ]:
%%task ch14/01-pixels/dark-rule
import numpy as np
import pandas as pd

### Средняя картинка класса · посложнее

Разбейте индексы на обучение и тест: `train_test_split(np.arange(len(X)), test_size=0.25, random_state=42, stratify=y)`. По обучающим картинкам посчитайте `means` — массив `(4, 32, 32)` средних картинок классов. Классифицируйте тестовые картинки по ближайшей средней (евклидово расстояние по всем пикселям) и сохраните `centroid_acc` — долю верных (3 знака). Нарисуйте четыре средние картинки.

*Подсказки (2): `dsc.hint('centroid')`*

In [ ]:
%%task ch14/01-pixels/centroid
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

### Сдвиг против смысла · со звёздочкой

Возьмите первый круассан `x0` (в `float`). Посчитайте `d_shift` — средние евклидовы расстояния от `x0` до его копий, сдвинутых `np.roll` по горизонтали на 1, 2, …, 6 пикселей, — словарь `{сдвиг: расстояние}` (целые, `round`). Посчитайте `d_same` — среднее расстояние до остальных круассанов и `d_empty` — до пустых подносов (целые). В `max_safe_shift` — наибольший сдвиг, при котором копия ещё ближе к `x0`, чем `d_empty`.

*Подсказки (3): `dsc.hint('ch14/01-pixels/shift')` · задача по желанию*

In [ ]:
%%task ch14/01-pixels/shift
import numpy as np

i = np.flatnonzero(y == 1)[0]
x0 = X[i]

### Из цвета в серое · база

Напишите `to_gray(rgb)`: для массива формы `(высота, ширина, 3)` типа `uint8` вернуть серую картинку `(высота, ширина)` типа `uint8` по формуле `0.299·R + 0.587·G + 0.114·B` с округлением. Примените её к `cup` из заготовки (картинка 6 × 6: пена, кофе и красный край стакана) — результат в `cup_gray`. В `darkest_color` — какой из трёх исходных цветов (`"пена"`, `"кофе"`, `"стакан"`) стал самым тёмным серым.

*Подсказки (3): `dsc.hint('gray')`*

In [ ]:
%%task ch14/01-pixels/gray
import numpy as np

def to_gray(rgb):
    ...

cup_gray = to_gray(cup)
print(cup_gray)

### Разрезать кадр на ячейки · база

В заготовке — кадр витрины `frame` 64 × 96 (2 ряда × 3 ячейки 32 × 32). Разрежьте его на ячейки: `cells` — список из 6 массивов 32 × 32 в порядке «слева направо, сверху вниз». Для каждой ячейки посчитайте долю пикселей темнее 100 — `dark_shares` (список, 3 знака) — и решите, пуста ли она (доля < 0.005): `empty` — список `True/False`. В `n_empty` — сколько пустых.

*Подсказки (3): `dsc.hint('cells')`*

In [ ]:
%%task ch14/01-pixels/cells
import numpy as np

cells = []

### Где лежит выпечка · посложнее

Насколько сильно выпечка «гуляет» по кадру? Для каждой **непустой** картинки найдите центр тёмного пятна: средний номер строки и средний номер столбца среди пикселей темнее 100. Сохраните `rows_c` и `cols_c` — массивы этих центров (по всем непустым картинкам, у которых есть хотя бы один тёмный пиксель), `spread` — кортеж из стандартных отклонений центров по строкам и столбцам (1 знак), и `frac_off_center` — долю картинок, у которых центр отстоит от середины кадра (15.5, 15.5) больше чем на 4 пикселя по какой-либо оси (2 знака).

*Подсказки (3): `dsc.hint('ch14/01-pixels/where')`*

In [ ]:
%%task ch14/01-pixels/where
import numpy as np

rows_c, cols_c = [], []

## Урок 2. Файлы картинок и память

Камера пишет тысячи цветных JPEG в день, а в памяти каждый кадр занимает столько, сколько в нём пикселей, как бы он ни был сжат на диске. Считаем память под кадры руками, сравниваем PNG и JPEG на кадрах витрины и разбираем ошибки, из-за которых ноутбук зависает на загрузке фото.

Теория — на сайте курса: `ch14/01b-files.html`.

### Сколько памяти нужно кадрам · разминка

Камера пишет цветные кадры 1920 × 1080 (3 канала). Посчитайте, сколько гигабайт (1 ГБ = 10⁹ байт) займут в памяти 10 000 кадров в `uint8` — `gb_uint8` — и в `float32` — `gb_float32` (оба — 3 знака). Затем посчитайте `frames_224` — сколько цветных кадров, уменьшенных до 224 × 224, поместится в 8 ГБ в `uint8` (целое число).

*Подсказки (2): `dsc.hint('memory')`*

In [ ]:
%%task ch14/01b-files/memory
import numpy as np
h, w, ch = 1080, 1920, 3
n = 10_000

### PNG против JPEG · база

Сохраните первые 100 кадров `X[:100]` в память (`io.BytesIO`) дважды — в PNG и в JPEG с `quality=75` — и прочитайте обратно. Посчитайте: `share_png` и `share_jpeg` — суммарный размер файлов как доля от `X[:100].nbytes` (3 знака), `mae_jpeg` — среднюю абсолютную ошибку пикселя после JPEG по всем 100 кадрам (2 знака) и `lossless` — `True`, если после PNG **все** кадры совпали с исходными до единого пикселя.

*Подсказки (3): `dsc.hint('formats')`*

In [ ]:
%%task ch14/01b-files/formats
import io
import numpy as np
from PIL import Image

X = np.load("showcase.npz")["X"]

def roundtrip(img, fmt, **opts):
    """Сохранить картинку в память и прочитать обратно: (размер в байтах, массив)."""
    buf = io.BytesIO()
    Image.fromarray(img).save(buf, format=fmt, **opts)
    data = buf.getvalue()
    return len(data), np.array(Image.open(io.BytesIO(data)))

### Серый и маленький · разминка

В `rgb` — цветной кадр витрины 64 × 96 при тёплом вечернем свете. Модель ждёт серые картинки вдвое меньше. С помощью Pillow переведите кадр в серый (`.convert("L")`) и уменьшите до ширины 48 и высоты 32. Результат как массив numpy сохраните в `small`, а в `ratio` — во сколько раз `small` занимает меньше памяти, чем `rgb` (через `.nbytes`).

*Подсказки (2): `dsc.hint('small')`*

In [ ]:
%%task ch14/01b-files/small
import numpy as np
from PIL import Image
# готово, как в теории: rgb — массив (64, 96, 3) типа uint8

### Пакеты вместо «всё сразу» · база

10 000 кадров уменьшены до 224 × 224, в цвете (3 канала), и подаются в сеть пакетами по 64 кадра в `float32`. Посчитайте `batch_mb` — сколько мегабайт (1 МБ = 10⁶ байт) занимает один пакет (1 знак), `n_batches` — сколько пакетов нужно на все кадры, `last_size` — сколько кадров в последнем, неполном пакете, и `all_gb` — сколько гигабайт заняли бы все 10 000 кадров в `float32` разом (2 знака).

*Подсказки (2): `dsc.hint('ch14/01b-files/batches')`*

In [ ]:
%%task ch14/01b-files/batches
import numpy as np
n, side, ch, batch = 10_000, 224, 3, 64

### Цена качества JPEG · база

Сохраните первые 100 кадров `X[:100]` в JPEG с качеством `[90, 75, 50, 30, 10]`. Для каждого качества посчитайте долю суммарного размера файлов от `X[:100].nbytes` и среднюю абсолютную ошибку пикселя. Сохраните `table` — DataFrame с индексом «качество» и столбцами `share` (3 знака) и `mae` (2 знака), а `q_half` — самое высокое качество, при котором файлы меньше половины исходного объёма.

*Подсказки (2): `dsc.hint('ch14/01b-files/quality')`*

In [ ]:
%%task ch14/01b-files/quality
import io
import numpy as np
import pandas as pd
from PIL import Image

X = np.load("showcase.npz")["X"]

def roundtrip(img, fmt, **opts):
    """Сохранить картинку в память и прочитать обратно: (размер в байтах, массив)."""
    buf = io.BytesIO()
    Image.fromarray(img).save(buf, format=fmt, **opts)
    data = buf.getvalue()
    return len(data), np.array(Image.open(io.BytesIO(data)))

### Маска в JPEG · посложнее

Маска выпечки — кадр, где тёмные пиксели (меньше 100) помечены 255, а остальные — 0. Для первых 100 кадров постройте маски и сохраните их в JPEG (`quality=90`) и в PNG. Посчитайте: `share_gray` — долю пикселей, которые после JPEG стали не 0 и не 255 (3 знака); `share_flipped` — долю пикселей, у которых после JPEG и порога `>= 128` поменялся класс (3 знака); `jpeg_share` и `png_share` — суммарный размер файлов JPEG и PNG как долю от объёма масок в памяти (3 знака).

*Подсказки (2): `dsc.hint('mask')`*

In [ ]:
%%task ch14/01b-files/mask
import io
import numpy as np
from PIL import Image

X = np.load("showcase.npz")["X"]
masks = ((X[:100] < 100) * 255).astype(np.uint8)

def roundtrip(img, fmt, **opts):
    """Сохранить картинку в память и прочитать обратно: (размер в байтах, массив)."""
    buf = io.BytesIO()
    Image.fromarray(img).save(buf, format=fmt, **opts)
    data = buf.getvalue()
    return len(data), np.array(Image.open(io.BytesIO(data)))

## Урок 3. Свёртка и фильтры

Свёртка — главная операция компьютерного зрения: маленькое окно-фильтр проходит по картинке и в каждом месте считает взвешенную сумму пикселей. Разные веса — разные фильтры: размытие убирает шум, фильтр Собеля находит края. Пишем свёртку руками, применяем готовую из scipy и собираем из фильтров и пулинга признаки, с которыми простая модель видит гораздо лучше.

Теория — на сайте курса: `ch14/02-filters.html`. Задачи «собери код из строк» (1) остались на сайте.

### Свёртка руками · разминка

Напишите функцию `conv2d(img, kernel)` — свёртку без дополнения краёв (как в первом примере урока, без переворота фильтра) для квадратного фильтра. Примените её к `img` из заготовки с фильтром `blur` — результат в `out`.

*Подсказки (3): `dsc.hint('ch14/02-filters/conv')`*

In [ ]:
%%task ch14/02-filters/conv
import numpy as np

def conv2d(img, kernel):
    ...

out = conv2d(img, blur)
print(out)

### Сила края по классам · база

Для всех картинок (`X` уже в долях 0–1) посчитайте карту силы края: размытие 3 × 3 (`BLUR`), затем Собель по x и y (`SX`, `SX.T`), `np.hypot`. Везде `ndi.correlate(..., mode="nearest")`. Для каждой картинки возьмите 95-й процентиль силы края (`np.percentile(..., 95)`) и сохраните `edge95` — массив длины 2000 и `edge_by_class` — Series средних `edge95` по названиям классов (3 знака). То же без размытия — `edge_by_class_raw`.

*Подсказки (3): `dsc.hint('edges')`*

In [ ]:
%%task ch14/02-filters/edges
import numpy as np
import pandas as pd
from scipy import ndimage as ndi

def edge_map(img, smooth=True):
    ...

### Пулинг для всего набора · база

Напишите `max_pool(F, s)` для массива карт формы `(n, h, w)` (h и w делятся на s): результат — `(n, h // s, w // s)`. Напишите и `avg_pool(F, s)` — то же со средним. Примените обе к `X` с `s=4`: `P_max`, `P_avg`. В `ratio` — во сколько раз уменьшилось число чисел на картинку.

*Подсказки (3): `dsc.hint('pool')`*

In [ ]:
%%task ch14/02-filters/pool
import numpy as np

def max_pool(F, s):
    ...

def avg_pool(F, s):
    ...

### Размытие против шума · посложнее

Шум удобно мерить на пустых подносах: там нет выпечки, и разброс яркости в центре кадра (`[4:28, 4:28]`) — это шум плюс плавный градиент подноса. Для пустых картинок посчитайте средний по картинкам `np.std` центра до размытия (`noise_raw`) и после размытия фильтрами 3 × 3 и 5 × 5 (`noise_3`, `noise_5`), все — 4 знака. Сохраните `reduction_3` — во сколько раз размытие 3 × 3 уменьшило разброс (2 знака).

*Подсказки (3): `dsc.hint('denoise')`*

In [ ]:
%%task ch14/02-filters/denoise
import numpy as np
from scipy import ndimage as ndi

empty = X[y == 0]

### Пиксели против фильтров · посложнее

Повторите эксперимент из урока: признаки `conv_features` (4 фильтра, модуль для первых трёх, max-pooling 4) и логистическая регрессия со `StandardScaler` (`max_iter=3000`). Разбиение — `train_test_split(np.arange(len(X)), test_size=0.25, random_state=42, stratify=y)`. Сохраните `acc_pixels`, `acc_filters` (3 знака) и `acc_pool2` — то же с пулингом 2 вместо 4.

Признаки для 2000 кадров считаются трижды, и в браузере это занимает около полуминуты — после запуска подождите, не нажимайте «Запустить» повторно.

*Подсказки (3): `dsc.hint('ch14/02-filters/features')`*

In [ ]:
%%task ch14/02-filters/features
import numpy as np
from scipy import ndimage as ndi
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

def conv_features(Z, s=4):
    ...

### Девять чисел вместо тысячи · со звёздочкой

Пулинг по всей картинке (**global pooling**) полностью убирает зависимость от положения. Для каждой картинки посчитайте 9 признаков: для фильтров `SX`, `SX.T`, `LAP` — среднее и максимум модуля отклика; для размытия `BLUR` — минимум, среднее и долю пикселей темнее 0.4. Сохраните `G` — массив `(2000, 9)` в этом порядке. `LAP` — лапласиан из раздела «Признаки из фильтров», он откликается на пятна и точки. Обучите `RandomForestClassifier(300, random_state=0)` на `G` и на сырых пикселях (разбиение как в задаче «Пиксели против фильтров»): `acc_raw`, `acc_global` (3 знака).

Лес из 300 деревьев на 1024 пикселях обучается в браузере до полуминуты — после запуска подождите.

*Подсказки (3): `dsc.hint('global')` · задача по желанию*

In [ ]:
%%task ch14/02-filters/global
import numpy as np
from scipy import ndimage as ndi
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

### Чем дополнять края · разминка

Для картинки `img` (светлый квадрат 6 × 6, все пиксели 0.8) примените Собель по x (`SX`) с тремя способами дополнения: `"nearest"`, `"reflect"`, `"constant"`. Сохраните `max_abs` — словарь `{режим: наибольший модуль отклика}` (2 знака) и `fake_edge_mode` — режим, который «нашёл» край на пустом ровном квадрате.

*Подсказки (2): `dsc.hint('modes')`*

In [ ]:
%%task ch14/02-filters/modes
import numpy as np
from scipy import ndimage as ndi

### Направление краёв · база

Повторите разбор из урока: для каждой картинки (`X` — в долях 0–1) размойте её (`BLUR`), посчитайте суммы модулей откликов Собеля по x (`vert`) и по y (`horiz`) и их отношение `horiz / vert`. Сохраните `ratio` — массив длины 2000, `ratio_by_class` — Series медиан по названиям классов (2 знака) и `most_horizontal` — класс с наибольшей медианой.

*Подсказки (3): `dsc.hint('orient')`*

In [ ]:
%%task ch14/02-filters/orient
import numpy as np
import pandas as pd
from scipy import ndimage as ndi

### Правило по краям · посложнее

Признак «сила краёв» тоже может отличать пустой поднос. Для каждой картинки посчитайте `strong` — долю пикселей, где сила края (размытие `BLUR`, затем `np.hypot` двух Собелей) больше 0.5. Правило: «товар есть», если `strong >= t`. Переберите `t` из `[0.005, 0.01, 0.02, 0.04]` и сохраните `acc_by_t` (3 знака), `best_t`, а в `vs_dark` — разность лучшей точности этого правила и правила по тёмным пикселям из урока «Картинка — это массив чисел» (`(X < 100/255)` доля ≥ 0.005), 3 знака.

*Подсказки (3): `dsc.hint('edge-rule')`*

In [ ]:
%%task ch14/02-filters/edge-rule
import numpy as np
from scipy import ndimage as ndi

## Урок 4. Модели на пикселях: ошибки, аугментация, устойчивость

Обучаем знакомые модели sklearn прямо на пикселях: логистическую регрессию, kNN, лес и полносвязную сеть. Смотрим, где они путаются, учим лес на отражённых и сдвинутых копиях кадров, а затем устраиваем стресс-тест: сдвиг выпечки и вечерний свет. Выясняется, что точность на тесте ничего не говорит об устойчивости.

Теория — на сайте курса: `ch14/03-classic.html`. Задачи «собери код из строк» (1) остались на сайте.

### Картинки в таблицу · разминка

Превратите `X` (форма `(2000, 32, 32)`) в таблицу `X2d` формы `(2000, 1024)`. Сохраните `n_features` — число признаков, и `back` — первую строку `X2d`, превращённую обратно в картинку 32 × 32.

*Подсказки (2): `dsc.hint('flat')`*

In [ ]:
%%task ch14/03-classic/flat
import numpy as np

### Кого с кем путают · база

Обучите `RandomForestClassifier(100, random_state=0)` на пикселях (разбиение по индексам `tr, te` дано в заготовке). Сохраните `cm` — DataFrame матрицы ошибок с названиями классов в индексе и столбцах, `per_class` — Series доли верных по каждому истинному классу (полнота, 3 знака), и `worst_pair` — кортеж `(истинный класс, ответ модели)` с наибольшим числом ошибок.

*Подсказки (3): `dsc.hint('confusion')`*

In [ ]:
%%task ch14/03-classic/confusion
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

### Куда смотрит лес · база

Обучите `RandomForestClassifier(100, random_state=0)` на пикселях обучающей части. Сложите `feature_importances_` в картинку `imp` 32 × 32. Сохраните `center_share` — долю важности в центре 16 × 16 (строки и столбцы 8…23), `border_share` — в рамке шириной 4 пикселя по краю кадра (всё, кроме `[4:28, 4:28]`), обе — 3 знака, и `top_pixel` — кортеж `(строка, столбец)` самого важного пикселя.

*Подсказки (3): `dsc.hint('ch14/03-classic/importance')`*

In [ ]:
%%task ch14/03-classic/importance
import numpy as np
from sklearn.ensemble import RandomForestClassifier

### Сколько соседей брать · база

Обучите `KNeighborsClassifier(k)` на пикселях для `k` из `[1, 3, 5, 15, 40]`. Сохраните `acc_by_k` — словарь `{k: доля верных на тесте}` (3 знака) и `best_k`. В `drop_1_to_40` — насколько упала точность от `k=1` до `k=40` (3 знака).

*Подсказки (2): `dsc.hint('knn-k')`*

In [ ]:
%%task ch14/03-classic/knn-k
from sklearn.neighbors import KNeighborsClassifier

### Аугментация · посложнее

Напишите `augment(Z, labels, rng)` как в уроке: исходные кадры + отражённые по горизонтали + сдвинутые на случайные `(dy, dx)` из `rng.integers(-4, 5, size=(len(Z), 2))`. Примените к обучающей части с `rng = np.random.default_rng(0)` и обучите лес `RandomForestClassifier(100, random_state=0)`. Сохраните `acc_base`, `acc_aug` (3 знака на обычном тесте) и `n_train_aug` — размер расширенной выборки.

Задача обучает несколько лесов на тысячах кадров — в браузере это занимает до минуты. После запуска подождите и не нажимайте «Запустить» повторно.

*Подсказки (3): `dsc.hint('augment')`*

In [ ]:
%%task ch14/03-classic/augment
import numpy as np
from sklearn.ensemble import RandomForestClassifier

def augment(Z, labels, rng):
    ...

### Стресс-тест сдвигом · посложнее

Сравните устойчивость к сдвигу трёх моделей: лес на пикселях, лес на пикселях с аугментацией (`X_aug`, `y_aug` — в заготовке) и лес на 9 глобальных признаках из урока «Свёртка и фильтры» (функция `global_features` — в заготовке). Все леса — `RandomForestClassifier(100, random_state=0)`. Тестовые кадры сдвиньте на 4 пикселя вправо: `np.roll(X[te], 4, axis=2)`. Сохраните `stress` — DataFrame с индексом `["пиксели", "пиксели + аугментация", "глобальные признаки"]` и столбцами `обычный`, `сдвиг` (доли верных, 3 знака).

Задача обучает несколько лесов на тысячах кадров — в браузере это занимает до минуты. После запуска подождите и не нажимайте «Запустить» повторно.

*Подсказки (3): `dsc.hint('stress')`*

In [ ]:
%%task ch14/03-classic/stress
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

X_shift = np.roll(X[te], 4, axis=2)

### Аугментация яркостью · посложнее

Нормировка — не единственное лекарство от вечернего света. Добавьте к обучающей части её копию, где каждый кадр умножен на свой случайный множитель из `rng.uniform(0.6, 1.1, size=len(tr))` (`rng = np.random.default_rng(0)`). Обучите `RandomForestClassifier(100, random_state=0)` на 3000 кадрах. Сохраните `acc_normal` и `acc_dim` — точность на обычном тесте и на тесте × 0.7 (3 знака), и `acc_dim_base` — точность на тесте × 0.7 у леса без аугментации.

Задача обучает несколько лесов на тысячах кадров — в браузере это занимает до минуты. После запуска подождите и не нажимайте «Запустить» повторно.

*Подсказки (3): `dsc.hint('bright-aug')`*

In [ ]:
%%task ch14/03-classic/bright-aug
import numpy as np
from sklearn.ensemble import RandomForestClassifier

rng = np.random.default_rng(0)

### Вечерний свет · со звёздочкой

Проверьте модели на «вечерних» кадрах: тест, умноженный на 0.7. Напишите `normalize(Z)` — для массива кадров `(n, 32, 32)` вернуть таблицу `(n, 1024)`, где каждая строка — кадр минус его среднее, делённый на его стандартное отклонение. Обучите `RandomForestClassifier(100, random_state=0)` на сырых пикселях и на нормированных. Сохраните `light` — словарь `{"сырые": (обычный, тусклый), "нормированные": (обычный, тусклый)}` с долями верных (3 знака).

Задача обучает несколько лесов на тысячах кадров — в браузере это занимает до минуты. После запуска подождите и не нажимайте «Запустить» повторно.

*Подсказки (3): `dsc.hint('ch14/03-classic/light')` · задача по желанию*

In [ ]:
%%task ch14/03-classic/light
import numpy as np
from sklearn.ensemble import RandomForestClassifier

def normalize(Z):
    ...

X_dim = X[te] * 0.7

## Урок 5. Свёрточные нейросети

Свёрточная сеть — это фильтры и пулинг из урока «Свёртка и фильтры», только веса фильтров она подбирает сама градиентным спуском, а слои складывает друг на друга. Разбираем, как через сеть проходит картинка и почему у неё так мало параметров, считаем рецептивное поле, проверяем, что даже случайные фильтры с пулингом дают сильные признаки, и пишем сеть на PyTorch для своего компьютера. Перенос обучения и задачи кроме классификации — в следующих уроках.

Теория — на сайте курса: `ch14/04-cnn.html`. Задачи «собери код из строк» (1) остались на сайте.

### Считаем параметры · разминка

Напишите `dense_params(n_in, n_out)` и `conv_params(k, c_in, c_out)`. Посчитайте `p_net` — число параметров сети из урока на PyTorch: три свёрточных слоя 3 × 3 (1 → 16, 16 → 32, 32 → 64) и полносвязный 64 → 4 (у пулинга, ReLU и Flatten параметров нет).

*Подсказки (2): `dsc.hint('params')`*

In [ ]:
%%task ch14/04-cnn/params
def dense_params(n_in, n_out):
    ...

def conv_params(k, c_in, c_out):
    ...

p_net = ...
print(p_net)

### Свёрточный слой · база

Напишите `conv_layer(img, W, b)`: `img` — картинка `(h, w)`, `W` — фильтры `(c_out, k, k)`, `b` — сдвиги `(c_out,)`. Вернуть массив `(c_out, h, w)`: для каждого фильтра `ndi.correlate(img, W[c], mode="nearest") + b[c]`, затем ReLU. Напишите `pool2(maps)` — max-pooling 2 × 2 для массива `(c, h, w)`. Примените к `X[0]` с весами `W1`, `b1` из заготовки: `out = pool2(conv_layer(X[0], W1, b1))`. В `active` — долю положительных чисел в `out` (3 знака).

*Подсказки (3): `dsc.hint('layer')`*

In [ ]:
%%task ch14/04-cnn/layer
import numpy as np
from scipy import ndimage as ndi

def conv_layer(img, W, b):
    ...

def pool2(maps):
    ...

### Голова сети · база

К выходу `out` (форма `(6, 16, 16)`, вычислен в заготовке) приделайте «голову»: вытянуть в вектор, умножить на `W2` (форма `(1536, 4)`), прибавить `b2`, применить softmax. Сохраните `logits`, `probs` (вероятности 4 классов, сумма 1) и `pred_class` — название класса с наибольшей вероятностью (из `classes`). Для устойчивости вычитайте максимум логита перед экспонентой.

*Подсказки (2): `dsc.hint('ch14/04-cnn/softmax')`*

In [ ]:
%%task ch14/04-cnn/softmax
import numpy as np

### Сколько случайных фильтров нужно · посложнее

Повторите эксперимент со случайными фильтрами (функция `random_conv_features` и разбиение `tr, te` — в заготовке) для 4, 8, 16 и 32 фильтров. Веса для каждого варианта — `np.random.default_rng(0).normal(size=(n, 5, 5))` (каждый раз новый генератор с seed 0). Модель — `StandardScaler` + `LogisticRegression(max_iter=3000)`. Сохраните `acc_by_n` — словарь `{n: доля верных}` (3 знака) и `n_enough` — наименьшее `n`, при котором доля верных выше 0.93.

Признаки случайных фильтров для 2000 кадров считаются в браузере до минуты — после запуска подождите и не нажимайте «Запустить» повторно.

*Подсказки (2): `dsc.hint('random')`*

In [ ]:
%%task ch14/04-cnn/random
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

### Случайные фильтры и вечерний свет · со звёздочкой

Проверьте признаки из 16 случайных фильтров на тусклых кадрах (`X[te] * 0.7`) — с нормировкой кадров перед фильтрами и без. Нормировка кадра: вычесть его среднее, разделить на его стандартное отклонение (форма `(n, 32, 32)` сохраняется). Модель — `StandardScaler` + `LogisticRegression(max_iter=3000)`, веса — `np.random.default_rng(0).normal(size=(16, 5, 5))`. Сохраните `res` — словарь `{"без нормировки": (обычный, тусклый), "с нормировкой": (обычный, тусклый)}` (3 знака).

Признаки случайных фильтров для 2000 кадров считаются в браузере до минуты — после запуска подождите и не нажимайте «Запустить» повторно.

*Подсказки (3): `dsc.hint('rand-light')` · задача по желанию*

In [ ]:
%%task ch14/04-cnn/rand-light
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

W = np.random.default_rng(0).normal(size=(16, 5, 5))

def norm_frames(Z):
    ...

### Размер выхода слоя · база

Напишите `out_size(n, k, pad=0, stride=1)` — размер стороны выхода свёртки (или пулинга) для входа `n × n`, окна `k`, дополнения `pad` с каждой стороны и шага `stride`: `(n + 2·pad − k) // stride + 1`. Посчитайте `sizes` — список размеров после каждого слоя сети из урока для входа 32: свёртка 3 (`pad=1`), пулинг 2 (`stride=2`), свёртка 3 (`pad=1`), пулинг 2 (`stride=2`), свёртка 3 (`pad=1`). В `no_pad` — то же, но все свёртки без дополнения.

*Подсказки (3): `dsc.hint('out-size')`*

In [ ]:
%%task ch14/04-cnn/out-size
def out_size(n, k, pad=0, stride=1):
    ...

### Рецептивное поле · база

Напишите `receptive_field(layers)` как в уроке: `layers` — список пар `(вид, k)`, где вид `"conv"` (свёртка k × k с шагом 1) или `"pool"` (пулинг k × k с шагом k). Посчитайте `rf_lesson` — поле для сети из урока, `rf_no_pool` — для пяти свёрток 3 подряд без пулингов и `n_convs_needed` — сколько свёрток 3 подряд (без пулингов) нужно, чтобы поле стало не меньше 32.

*Подсказки (3): `dsc.hint('receptive')`*

In [ ]:
%%task ch14/04-cnn/receptive
def receptive_field(layers):
    rf, jump = 1, 1
    ...
    return rf

### Пулинг против сдвига · посложнее

Повторите разбор из урока для пулинга `s` из `[4, 8, 32]` на 16 случайных фильтрах (функция `features(Z, s)` и разбиение — в заготовке). Сохраните `res` — DataFrame с индексом `s` и столбцами `n_features`, `test`, `shifted` (3 знака; сдвиг — `np.roll(X[te], 4, axis=2)`), `best_test` — `s` с лучшей точностью на обычном тесте и `most_robust` — `s` с наименьшим падением от сдвига.

Признаки случайных фильтров для 2000 кадров считаются в браузере до минуты — после запуска подождите и не нажимайте «Запустить» повторно.

*Подсказки (3): `dsc.hint('pool-size')`*

In [ ]:
%%task ch14/04-cnn/pool-size
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_shift = np.roll(X[te], 4, axis=2)
rows = []

## Урок 6. Перенос обучения и дообучение

Для обычных фотографий сеть с нуля почти не учат: берут готовую, обученную на миллионе снимков, и подстраивают под свои классы. Разбираем два способа — признаки готовой сети и дообучение по шагам, — загружаем фото из папок классов, где попадаются мусорные и битые файлы, и смотрим, почему ResNet может быть такой глубокой.

Теория — на сайте курса: `ch14/04b-transfer.html`.

### Загрузить фото из папок · база

В заготовке на диске создана папка `vitrina/train` с подпапками классов — как приходят фото от фотографа. Кроме картинок там есть мусор: служебные файлы систем, заметки, один битый PNG, а одна фотография сохранена с расширением в верхнем регистре (`.PNG`). Загрузите все **читаемые** PNG (расширение без учёта регистра) в массив `X` формы (n, 32, 32), метки — в `y` (номер класса по алфавиту имён папок, как у `ImageFolder`), имена классов — в `classes` (список по алфавиту). Сохраните `counts` — словарь «класс → число фото» и `bad` — список имён файлов (`path.name`), которые не удалось прочитать.

*Подсказки (3): `dsc.hint('folders')`*

In [ ]:
%%task ch14/04b-transfer/folders
from pathlib import Path
import numpy as np
from PIL import Image

root = Path("vitrina/train")
print(sorted(p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file())[:6])

### Голова под свои классы · разминка

Напишите `head_params(n_in, n_classes)` — сколько параметров у новой головы `nn.Linear(n_in, n_classes)`: вес на каждую пару «вход — класс» и сдвиг на каждый класс. Сохраните `p_resnet18` — голову ResNet-18 (512 входов) под 4 класса витрины и `p_resnet50` — голову ResNet-50 (2048 входов) под 10 классов.

*Подсказки (2): `dsc.hint('head')`*

In [ ]:
%%task ch14/04b-transfer/head
def head_params(n_in, n_classes):
    ...

### Что учится на каждом шаге · база

В `LAYERS` — число параметров в каждой части ResNet-18 с новой головой на 4 класса (так их печатает `sum(p.numel() for p in model.layer4.parameters())` и т. п.). Посчитайте `total` — всего параметров, `step2` — сколько обучается на шаге 2 (только голова `fc`), `step3` — на шаге 3 (`fc` и `layer4`) и `share3` — долю обучаемых на шаге 3 от всех (3 знака).

*Подсказки (2): `dsc.hint('trainable')`*

In [ ]:
%%task ch14/04b-transfer/trainable
# готово, как в условии: LAYERS — словарь {часть сети: число параметров}

### Что забыли заморозить · база

Коллега выполнил шаг 2 дообучения и жалуется, что сеть быстро переобучается. Он прислал состояние частей сети: `STATE` — словарь `{часть: (число параметров, обучается ли)}`. Посчитайте `trainable` — сколько параметров сейчас обучается, `unfrozen` — список частей, которые обучаются, хотя на шаге 2 не должны (всё, кроме `fc`, в порядке словаря), и `extra` — во сколько раз обучаемых параметров больше, чем положено на шаге 2 (целое, округлите).

*Подсказки (2): `dsc.hint('frozen-bug')`*

In [ ]:
%%task ch14/04b-transfer/frozen-bug
# готово, как в условии: STATE — словарь {часть: (параметров, обучается ли)}

### Сколько фото на класс · посложнее

В `E` — признаки всех 2000 кадров из «замороженного тела»: 16 случайных фильтров 5 × 5 и пулинг 8, как в уроке «Свёрточные нейросети» (`E.shape` — `(2000, 256)`). Разбиение `tr, te` — как во всей главе. Для `n` из `[10, 25, 50, 100, 200]` обучите `StandardScaler` + `LogisticRegression(max_iter=3000)` на первых `n` обучающих кадрах **каждого** класса (по порядку `tr`) и посчитайте точность на всём тесте. Сохраните `curve` — словарь `{n: точность}` (3 знака) и `n_enough` — наименьшее `n`, у которого точность отстаёт от точности при 200 не больше чем на 0.02.

Подготовка признаков для 2000 кадров в браузере занимает до полуминуты — после запуска подождите.

*Подсказки (2): `dsc.hint('few-shot')`*

In [ ]:
%%task ch14/04b-transfer/few-shot
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
# готово, как в условии: E (признаки), y, tr, te

## Урок 7. Регрессия по фото, детекция и сегментация

Сеть умеет не только называть класс. Она может оценить по фото число — вес порции или заполненность подноса, найти на кадре все объекты рамками (детекция) или отметить их пиксели (сегментация). Разбираем, что меняется в сети для каждой задачи, и главное для аналитика — как проверять результат: IoU, сопоставление рамок, точность и полнота детектора.

Теория — на сайте курса: `ch14/04c-detection.html`.

### Пересечение на объединение · база

Напишите функцию `iou(a, b)` для рамок `(x1, y1, x2, y2)`. В заготовке — шесть настоящих рамок выпечки на подносе и шесть рамок модели (по порядку: `pred[i]` — попытка найти `true[i]`). Сохраните `ious` — список IoU для каждой пары (3 знака) и `n_hits` — сколько пар с IoU не ниже 0.5.

*Подсказки (3): `dsc.hint('iou')`*

In [ ]:
%%task ch14/04c-detection/iou
true = [(2, 3, 30, 29), (34, 2, 62, 30), (66, 4, 94, 31), (1, 34, 29, 62), (33, 35, 63, 61), (65, 33, 95, 63)]
pred = [(4, 5, 31, 30), (40, 8, 62, 30), (66, 4, 94, 31), (10, 40, 29, 62), (30, 30, 60, 60), (80, 45, 95, 63)]

def iou(a, b):
    ...

### Точность и полнота детектора · посложнее

Модель нашла на кадре 7 рамок с уверенностью (`dets`), настоящих рамок 6 (`true`). Сопоставьте их, как это делают при оценке детекторов: идите по найденным рамкам **по убыванию уверенности**; для каждой найдите ещё не занятую настоящую рамку с наибольшим IoU; если IoU ≥ 0.5 — это попадание (настоящая рамка становится занятой), иначе — ложное срабатывание. Настоящие рамки, оставшиеся незанятыми, — пропуски. Сохраните `tp`, `fp`, `fn`, `precision` и `recall` (3 знака). Функция `iou` готова.

*Подсказки (3): `dsc.hint('match')`*

In [ ]:
%%task ch14/04c-detection/match
# уже есть: iou(a, b)
true = [(2, 3, 30, 29), (34, 2, 62, 30), (66, 4, 94, 31), (1, 34, 29, 62), (33, 35, 63, 61), (65, 33, 95, 63)]
dets = [((4, 2, 31, 28), 0.81), ((66, 5, 93, 31), 0.95), ((35, 36, 62, 60), 0.77), ((3, 4, 30, 30), 0.97),
        ((70, 40, 90, 60), 0.41), ((34, 3, 50, 29), 0.64), ((60, 0, 96, 10), 0.30)]

### IoU одной пары · разминка

Настоящая рамка `a = (0, 0, 10, 10)`, найденная `b = (5, 0, 15, 10)`. Посчитайте по шагам, как в примере урока: `inter` — площадь пересечения, `union` — площадь объединения и `iou_ab` — их отношение (3 знака). Пишите формулы через координаты, а не готовые числа.

*Подсказки (2): `dsc.hint('iou-hand')`*

In [ ]:
%%task ch14/04c-detection/iou-hand
a = (0, 0, 10, 10)
b = (5, 0, 15, 10)

### Рамки в разных форматах · база

Разметчик сохранил настоящие рамки в формате COCO — `(x, y, ширина, высота)`, а модель выдаёт рамки в формате YOLO — `(центр x, центр y, ширина, высота)` в долях от размера кадра 96 × 64. Напишите `coco_to_xyxy(b)` и `yolo_to_xyxy(b, W, H)`, переводящие рамку в `(x1, y1, x2, y2)` в пикселях, и сохраните `ious` — IoU пар `pred_yolo[i]` и `true_coco[i]` после перевода (3 знака). Функция `iou` для рамок `(x1, y1, x2, y2)` готова.

*Подсказки (2): `dsc.hint('box-format')`*

In [ ]:
%%task ch14/04c-detection/box-format
# уже есть: iou(a, b) для рамок (x1, y1, x2, y2)
W, H = 96, 64
true_coco = [(2, 3, 28, 26), (34, 2, 28, 28), (66, 4, 28, 27)]
pred_yolo = [(0.1823, 0.2734, 0.2812, 0.3906), (0.5312, 0.2969, 0.2292, 0.3438), (0.8333, 0.2734, 0.2917, 0.4219)]

def coco_to_xyxy(b):
    ...

def yolo_to_xyxy(b, W, H):
    ...

### Маска из вероятностей · база

Модель сегментации выдаёт для каждого пикселя вероятность «здесь выпечка» — массив `prob` 32 × 32. Маска получается порогом: `prob > t`. Настоящая маска — `true_mask`. Для порогов `[0.3, 0.5, 0.7]` посчитайте IoU маски модели с настоящей и сохраните `iou_by_t` — словарь `{порог: IoU}` (3 знака), `best_t` — порог с наибольшим IoU, а также `occ_true` и `occ_best` — долю кадра, занятую выпечкой, по настоящей маске и по маске модели при лучшем пороге (3 знака).

*Подсказки (2): `dsc.hint('mask-iou')`*

In [ ]:
%%task ch14/04c-detection/mask-iou
import numpy as np
# готово, как в условии: prob (вероятности), true_mask (булева маска)

### Убрать дубли рамок · со звёздочкой

Напишите `nms(dets, thr=0.5)` — подавление дублей, как в уроке: перебирайте рамки по убыванию уверенности; рамку оставляйте, если её IoU с **каждой** уже оставленной не больше `thr`, иначе выбрасывайте. Функция возвращает список оставленных пар `(рамка, уверенность)` в порядке убывания уверенности. Примените её к `dets` из задачи «Точность и полнота детектора»: `kept_conf` — список уверенностей оставленных рамок. Затем сопоставьте оставленные рамки с настоящими, как в той задаче (функция `match` готова), и сохраните `precision` и `recall` (3 знака).

*Подсказки (2): `dsc.hint('nms')` · задача по желанию*

In [ ]:
%%task ch14/04c-detection/nms
# уже есть: iou(a, b) и match(dets, true) → (tp, fp, fn)
true = [(2, 3, 30, 29), (34, 2, 62, 30), (66, 4, 94, 31), (1, 34, 29, 62), (33, 35, 63, 61), (65, 33, 95, 63)]
dets = [((4, 2, 31, 28), 0.81), ((66, 5, 93, 31), 0.95), ((35, 36, 62, 60), 0.77), ((3, 4, 30, 30), 0.97),
        ((70, 40, 90, 60), 0.41), ((34, 3, 50, 29), 0.64), ((60, 0, 96, 10), 0.30)]

def nms(dets, thr=0.5):
    ...

## Урок 8. Проект: что закончилось на витрине

Финал главы. Камера над витриной снимает 12 ячеек каждые 15 минут, и пекарня хочет получать сигнал, когда выпечка закончилась. Модель, обученная на дневных кадрах, отлично работает днём — и полностью слепнет вечером, когда в зале включают другой свет. Находим причину, чиним, строим правило оповещений и отвечаем пекарне, когда допекать.

Теория — на сайте курса: `ch14/05-project.html`.

### Шаг 1. День и вечер · база

Обучите `RandomForestClassifier(100, random_state=0)` на всех 2000 вырезках (`X / 255`, вытянутые в строки). Предскажите классы для кадров дня (`XD / 255`). Сохраните `pred_raw`, `acc_day` и `acc_evening` — доли верных до 18:00 и с 18:00 (3 знака), `n_empty_evening` — сколько вечерних кадров на самом деле пустые и `n_found_evening` — сколько из них модель назвала пустыми.

*Подсказки (2): `dsc.hint('day')`*

In [ ]:
%%task ch14/05-project/day
import numpy as np
from sklearn.ensemble import RandomForestClassifier

d = np.load("showcase.npz")
X, y = d["X"] / 255, d["y"]
D = np.load("showcase_day.npz")
XD, slot, tm, yD = D["X"] / 255, D["slot"], D["time"], D["y"]
flat = lambda Z: Z.reshape(len(Z), -1)

### Шаг 2. Что изменилось · база

Посчитайте среднюю яркость каждого кадра дня и усредните по времени: `bright` — Series с индексом-временем (61 значение). Сохраните `ratio` — отношение средней яркости вечерних кадров (с 18:00) к дневным (3 знака) и `step_time` — время, в которое яркость упала сильнее всего относительно предыдущего момента. Нарисуйте `bright`.

*Подсказки (2): `dsc.hint('ch14/05-project/light')`*

In [ ]:
%%task ch14/05-project/light
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

### Шаг 3. Нормировка · посложнее

Обучите такой же лес на нормированных вырезках: каждая строка (кадр) минус её среднее, делённое на её стандартное отклонение. Предскажите нормированные кадры дня: `pred_norm`. Сохраните `acc_day_n`, `acc_evening_n` (3 знака) и для вечера — `recall_empty` (доля пустых ячеек, которые модель нашла) и `precision_empty` (доля верных среди ответов «пусто»), 3 знака.

*Подсказки (2): `dsc.hint('fix')`*

In [ ]:
%%task ch14/05-project/fix
import numpy as np
from sklearn.ensemble import RandomForestClassifier

def normalize(Z):
    ...

### Шаг 4. Часы пустой витрины · база

Ответы нормированной модели — в `pred` (заготовка). Соберите таблицу `df` с колонками `time`, `slot`, `product` (название товара ячейки: 0–3 круассан, 4–7 маффин, 8–11 печенье), `true`, `pred`. Посчитайте для утра (до 13:00, когда допекают) сколько часов ячейки каждого товара стояли пустыми: каждый пустой кадр — 0.25 часа. Сохраните `empty_hours` — DataFrame с индексом-товаром и столбцами `модель` и `факт`.

*Подсказки (2): `dsc.hint('ch14/05-project/hours')`*

In [ ]:
%%task ch14/05-project/hours
import numpy as np
import pandas as pd

product = np.array(["круассан"] * 4 + ["маффин"] * 4 + ["печенье"] * 4)

### Шаг 5. Правило оповещений · посложнее

Одиночная ошибка модели не должна будить пекаря. Правило: оповещение по ячейке в момент `t`, когда ячейка пуста по модели второй кадр подряд — «пусто» в `t` и в `t − 15 мин`, но не в `t − 30 мин`. Так на каждую серию пустых кадров приходится одно оповещение, в её начале, а не по оповещению на каждый следующий кадр. Если серию прервал хотя бы один кадр «не пусто», следующая серия даст новое оповещение. Поэтому если модель внутри настоящей серии пустоты на один кадр ошибётся, правило даст повторное оповещение по той же ячейке — это нормально, и оповещений может оказаться больше, чем эпизодов пустоты. Таблица `df` — в заготовке.

Постройте `P` и `T` — таблицы «время × ячейка» из `True/False`: пусто по модели и по факту. Сохраните `alerts` — DataFrame с колонками `time`, `slot` (по одной строке на оповещение), `n_alerts`, `n_correct` — сколько оповещений пришлось на действительно пустую ячейку, и `n_missed` — сколько эпизодов пустоты по факту (серий подряд идущих пустых кадров ячейки) прошли без единого оповещения.

*Подсказки (2): `dsc.hint('alerts')`*

In [ ]:
%%task ch14/05-project/alerts
import numpy as np
import pandas as pd

P = df.pivot(index="time", columns="slot", values="pred") == 0
T = df.pivot(index="time", columns="slot", values="true") == 0

### Шаг 6. Когда допекать · со звёздочкой

По фактическим данным (`T` и `product` — в заготовке) найдите для каждого товара `half_empty` — первое время `t`, когда в момент `t` пусты не меньше 2 из 4 ячеек товара (Series с индексом-товаром). Посчитайте `saved_hours` — сколько часов пустых ячеек (по факту, 0.25 ч за кадр) приходится на интервал 11:00–12:45 включительно: столько простоя убрало бы допекание в 11:00 вместо 13:00 (при условии, что пекарня успеет). И `saved_by_product` — то же по товарам (Series).

*Подсказки (3): `dsc.hint('restock')` · задача по желанию*

In [ ]:
%%task ch14/05-project/restock
import numpy as np
import pandas as pd

## Готово

Запустите `dsc.progress()` — увидите, какие задачи этого сеанса ещё не решены. Перед тем как закрыть блокнот, полезно сделать **Kernel → Restart Kernel and Run All Cells**: если всё проходит сверху вниз, решения не зависят от случайного порядка запуска ячеек.

In [ ]:
dsc.progress()